In [57]:
import json
import pandas as pd
import numpy as np
import os

from pathlib import Path

# Step 2

## Overview
This Jupyter Notebook modifies configuration files for training SLEAP models. It reads a CSV file containing train, validation, and test splits, updates the initial configuration file with the appropriate paths for each split, and saves the modified configuration files.

## Prerequisites
Ensure the working directory contains the following files:
- `train_test_splits.csv`: A CSV file with columns `path`, `version`, `labeled_frames`, and `split_type` output from the previous notebook (`make_train_test_splits`).
- `initial_config.json`: A JSON configuration file to be modified based on the splits (add a config file that makes sense for your model).

## User-Defined Inputs
- `working_dir`: The directory where the CSV and JSON files are located.

## Steps
1. **Import Libraries**: Import necessary libraries.
2. **Define Working Directory**: Set the `working_dir` variable.
3. **Load CSV File**: Read the `train_test_splits.csv` file into a DataFrame (`splits`).
4. **Load JSON File**: Read the `initial_config.json` file into a dictionary (`data`).
5. **Modify Configuration**: Update the JSON data with the appropriate paths for each split type (train, val, test).
6. **Save Modified Configurations**: Save the modified JSON data to new files in the corresponding directories.

## Output
The final output is a set of modified configuration files saved in the directories specified in the `path` column of the CSV file. Each modified configuration file is named `initial_config_modified_v00X.json`, where `X` is the version number from the CSV file.

## Usage
1. Ensure the working directory contains the required `train_test_splits.csv` and `initial_config.json` files. These are the outputs from step 1.
2. Update the `working_dir` variable with the path to your working directory. This should be the `output_path` from step 1.
3. Use `Run All` to process the data and generate the modified configuration files.

In [58]:
# This should be the output path from the previous step
# working_dir = "D:/SLEAP/20250415_primary_root_generalizability_experiment/arabidopsis"
# working_dir = "D:/SLEAP/20250415_primary_root_generalizability_experiment/canola"
# working_dir = "D:/SLEAP/20250415_primary_root_generalizability_experiment/pennycress"
# working_dir = "D:/SLEAP/20250415_primary_root_generalizability_experiment/sorghum"
# working_dir = "D:/SLEAP/20250415_primary_root_generalizability_experiment/sorghum_soybean_canola_pennycress_rice_arabidopsis"
# working_dir = "D:/SLEAP/20250415_primary_root_generalizability_experiment/soybean"
working_dir = "D:/SLEAP/20250415_primary_root_generalizability_experiment/younger_rice"

In [59]:
# Since we are changing directories here, we need to set the base directory
old_base_dir = Path("D:/SLEAP/20250102_generalizability_experiment/primary")
new_base_dir = Path("D:/SLEAP/20250415_primary_root_generalizability_experiment")

In [60]:
# Load the CSV file from the previous step
csv_path = Path(working_dir) / "train_test_splits.csv"
splits = pd.read_csv(csv_path )
splits

,path,version,labeled_frames,split_type
0,D:\SLEAP\20250102_generalizability_experiment\...,0,737,train
1,D:\SLEAP\20250102_generalizability_experiment\...,0,158,val
2,D:\SLEAP\20250102_generalizability_experiment\...,0,158,test
3,D:\SLEAP\20250102_generalizability_experiment\...,1,737,train
4,D:\SLEAP\20250102_generalizability_experiment\...,1,158,val
5,D:\SLEAP\20250102_generalizability_experiment\...,1,158,test
6,D:\SLEAP\20250102_generalizability_experiment\...,2,737,train
7,D:\SLEAP\20250102_generalizability_experiment\...,2,158,val
8,D:\SLEAP\20250102_generalizability_experiment\...,2,158,test


In [61]:
# Path to the initial configuration file for the models 
# This is a template that will be modified for each model 
init_config_path = Path(working_dir) / "initial_config.json"

In [62]:
# Load the initial config JSON
with init_config_path.open('r') as file:
    data = json.load(file)

# Iterate over the rows of the DataFrame
for index, row in splits.iterrows():
    split_type = row['split_type']
    original_path = Path(row['path'])
    
    # Update label paths
    if split_type == 'train':
        data['data']['labels']['training_labels'] = original_path.as_posix()
    elif split_type == 'val':
        data['data']['labels']['validation_labels'] = original_path.as_posix()
    elif split_type == 'test':
        data['data']['labels']['test_labels'] = original_path.as_posix()

    # Compute the relative path and new config directory
    relative_split_path = original_path.parent.relative_to(old_base_dir)
    config_dir = new_base_dir / relative_split_path
    config_dir.mkdir(parents=True, exist_ok=True)
    # print(f"Config directory: {config_dir}")

    # Update output path
    data['outputs']['runs_folder'] = (config_dir / 'models').as_posix()

    # Save updated config
    new_config_path = config_dir / f"initial_config_modified_v00{row['version']}.json"
    with new_config_path.open('w') as new_file:
        json.dump(data, new_file)
        print(f"Saved modified config for split {split_type} version {row['version']} to {new_config_path}")

Saved modified config for split train version 0 to D:\SLEAP\20250415_primary_root_generalizability_experiment\younger_rice\train_test_split.v000\initial_config_modified_v000.json
Saved modified config for split val version 0 to D:\SLEAP\20250415_primary_root_generalizability_experiment\younger_rice\train_test_split.v000\initial_config_modified_v000.json
Saved modified config for split test version 0 to D:\SLEAP\20250415_primary_root_generalizability_experiment\younger_rice\train_test_split.v000\initial_config_modified_v000.json
Saved modified config for split train version 1 to D:\SLEAP\20250415_primary_root_generalizability_experiment\younger_rice\train_test_split.v001\initial_config_modified_v001.json
Saved modified config for split val version 1 to D:\SLEAP\20250415_primary_root_generalizability_experiment\younger_rice\train_test_split.v001\initial_config_modified_v001.json
Saved modified config for split test version 1 to D:\SLEAP\20250415_primary_root_generalizability_experiment\y

In [ ]:
# better to do this in the future to update labels paths in the config

# # Load the initial config JSON
# with init_config_path.open('r') as file:
#     data = json.load(file)

# # Iterate over the rows of the DataFrame
# for index, row in splits.iterrows():
#     split_type = row['split_type']
#     original_path = Path(row['path'])

#     # Compute the relative path and new config directory
#     relative_split_path = original_path.parent.relative_to(old_base_dir)
#     config_dir = new_base_dir / relative_split_path
#     config_dir.mkdir(parents=True, exist_ok=True)

#     # Compute new label path (same filename, new parent)
#     new_label_path = config_dir / original_path.name

#     # Update label paths with new base directory
#     if split_type == 'train':
#         data['data']['labels']['training_labels'] = new_label_path.as_posix()
#     elif split_type == 'val':
#         data['data']['labels']['validation_labels'] = new_label_path.as_posix()
#     elif split_type == 'test':
#         data['data']['labels']['test_labels'] = new_label_path.as_posix()

#     # Update output path
#     data['outputs']['runs_folder'] = (config_dir / 'models').as_posix()

#     # Save updated config
#     new_config_path = config_dir / f"initial_config_modified_v00{row['version']}.json"
#     with new_config_path.open('w') as new_file:
#         json.dump(data, new_file, indent=2)
#         print(f"Saved modified config for split {split_type} version {row['version']} to {new_config_path}")
